In [1]:
!pip install -q sentence-transformers

In [2]:
import pandas as pd
import torch
import seaborn as sns  # Added import
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from google.colab import drive

In [3]:
torch.cuda.empty_cache()

In [4]:
# 1. Setup & Load Data
drive.mount('/content/drive')
baseline_path = '/content/drive/MyDrive/Project/results/baseline_results.csv'
perturbed_path = '/content/drive/MyDrive/Project/results/perturbed_results.csv'

df_base = pd.read_csv(baseline_path)
df_pert = pd.read_csv(perturbed_path)

# FIX: Added 'perturbed_code' to the merge columns
df_merged = pd.merge(
    df_base[['index', 'cwe', 'baseline_explanation']], 
    df_pert[['index', 'perturbed_explanation', 'perturbed_code']], 
    on='index'
)

# 2. Load SBERT Model 'all-MiniLM-L6-v2'
model = SentenceTransformer('all-MiniLM-L6-v2')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def calculate_metrics(row):
    emb1 = model.encode(row['baseline_explanation'], convert_to_tensor=True)
    emb2 = model.encode(row['perturbed_explanation'], convert_to_tensor=True)
    cosine_sim = util.pytorch_cos_sim(emb1, emb2).item()
    return pd.Series([cosine_sim, 1 - cosine_sim])

print(f"--- Processing {len(df_merged)} samples for Semantic Instability ---")
df_merged[['cosine_similarity', 'semantic_variance']] = df_merged.apply(calculate_metrics, axis=1)

In [ ]:
# 3. Structural Change Detection (Workflow Insights)
# We identify if the perturbation was 'Renaming Only' or 'Structural (While/Ternary)'
def identify_perturbation_type(code):
    if 'while' in str(code).lower() and 'for' not in str(code).lower():
        return 'Structural (For->While)'
    elif '?' in str(code) and ':' in str(code):
        return 'Structural (Ternary)'
    else:
        return 'Lexical (Renaming Only)'

df_merged['perturbation_type'] = df_merged['perturbed_code'].apply(identify_perturbation_type)

In [ ]:
# 4. CWE Distribution & Top 3 Unstable CWEs
cwe_stats = df_merged.groupby('cwe')['semantic_variance'].agg(['mean', 'std', 'count']).sort_values(by='mean', ascending=False)

print("\n--- CRITIQUE ADDRESS: Top 5 Most Unstable CWEs ---")
top_5 = cwe_stats.head(5)
for cwe, row in top_5.iterrows():
    print(f"{cwe}: Mean Variance {row['mean']:.4f} (n={int(row['count'])})")

# 5. Workflow Impact Analysis
workflow_impact = df_merged.groupby('perturbation_type')['semantic_variance'].mean()
print("\n--- Workflow Insight: Mean Variance by Transformation ---")
print(workflow_impact)

# 6. Visualization: The "Brittleness" Dashboard
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

In [ ]:
# Plot A: CWE Instability with Sample Size
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(cwe_stats)))
sns.barplot(x=cwe_stats.index, y=cwe_stats['mean'], ax=ax1, palette="flare")
ax1.set_title("Semantic Instability by CWE Type", fontsize=14, fontweight='bold')
ax1.set_ylabel("Mean Variance (1 - Cosine Sim)")
ax1.tick_params(axis='x', rotation=45)

In [ ]:
# Plot B: Impact of Structural vs Lexical changes
sns.boxplot(x='perturbation_type', y='semantic_variance', data=df_merged, ax=ax2, palette="Set2")
ax2.set_title("Impact of Structural vs. Lexical Perturbations on Reasoning", fontsize=14, fontweight='bold')
ax2.set_ylabel("Semantic Variance")

plt.tight_layout()
plt.savefig(results_dir + "rq1_enhanced_insights.png", dpi=300)
plt.show()

# Save for methodology inclusion
df_merged.to_csv(results_dir + 'rq1_detailed_analysis.csv', index=False)